#1. 📦 INSTALL DEPENDENCIES (RUN FIRST)

In [10]:
!pip install -q \
langchain-core langchain-community langchain-huggingface langchain-text-splitters \
huggingface_hub pypdf faiss-cpu sentence-transformers gradio

print("Done!")

Done!


#2. 🔐 SET HUGGINGFACE TOKEN

In [11]:
import os
from getpass import getpass

os.environ['HUGGINGFACEHUB_API_TOKEN'] = getpass("HuggingFace API Token: ")
print("Token saved!")

HuggingFace API Token: ··········
Token saved!


#3. ⚙️ IMPORTS + CONFIGURATION

In [12]:
import os, time, warnings
warnings.filterwarnings('ignore')

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from huggingface_hub import InferenceClient
from pypdf import PdfReader
import gradio as gr

# Models
MODEL_MAP = {
    'Qwen2.5-7B': 'Qwen/Qwen2.5-7B-Instruct',
}

EMBED_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'

# Global state
vector_store = None
hf_client = None
active_model = None

#4. 📄 PDF LOADING + SPLITTING

In [13]:
def load_and_split(pdf_path, chunk_size=1000, chunk_overlap=150):
    pages = PyPDFLoader(pdf_path).load()

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=['\n\n', '\n', '. ', ' ', '']
    )

    return splitter.split_documents(pages)


def pdf_info(pdf_path):
    reader = PdfReader(pdf_path)
    return {
        "pages": len(reader.pages),
        "title": reader.metadata.get("/Title", "Unknown")
    }

#5. 🧠 EMBEDDINGS + VECTOR DATABASE (FAISS)

In [14]:
def embed(chunks):
    embeddings = HuggingFaceEmbeddings(
        model_name=EMBED_MODEL,
        model_kwargs={'device': 'cpu'},
        encode_kwargs={'normalize_embeddings': True}
    )

    return FAISS.from_documents(chunks, embeddings)

#🤖 LLM RESPONSE (QWEN API CALL)

In [15]:
def ask_llm(context, question):
    system = (
        "You are a helpful assistant. "
        "Answer ONLY using the provided PDF context. "
        "If not found, say you don't know."
    )

    user = f"""
PDF Context:
{context}

Question: {question}

Answer:
"""

    response = hf_client.chat_completion(
        model=active_model,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user}
        ],
        max_tokens=512,
        temperature=0.3,
    )

    return response.choices[0].message.content.strip()

#7. 🔄 PDF PROCESSING PIPELINE

In [16]:
def process_pdf(pdf_file, chunk_size, chunk_overlap, model_choice):
    global vector_store, hf_client, active_model

    try:
        start = time.time()

        path = pdf_file.name if hasattr(pdf_file, "name") else pdf_file
        info = pdf_info(path)

        print("Splitting PDF...")
        chunks = load_and_split(path, int(chunk_size), int(chunk_overlap))

        print("Creating embeddings...")
        vector_store = embed(chunks)

        hf_client = InferenceClient(
            token=os.environ["HUGGINGFACEHUB_API_TOKEN"]
        )

        active_model = MODEL_MAP[model_choice]

        elapsed = round(time.time() - start, 1)

        return (
            f"Ready in {elapsed}s\n"
            f"Pages: {info['pages']}\n"
            f"Chunks: {len(chunks)}\n"
            f"Model: {model_choice}\n"
            f"Title: {info['title']}\n\n"
            "Ask your questions below!"
        )

    except Exception as e:
        return f"Error: {e}"

#8. 💬 QUESTION ANSWERING (RAG LOGIC)

In [17]:
def answer_question(question, history):
    if vector_store is None:
        return "Please upload and process a PDF first."

    if not question.strip():
        return "Enter a valid question."

    try:
        docs = vector_store.similarity_search(question, k=4)

        context = "\n\n".join(d.page_content for d in docs)

        answer = ask_llm(context, question)

        pages = sorted({
            d.metadata.get("page")
            for d in docs
            if isinstance(d.metadata.get("page"), int)
        })

        if pages:
            answer += "\n\n(Sources: " + ", ".join(f"p.{p+1}" for p in pages) + ")"

        return answer

    except Exception as e:
        return f"Error: {e}"

#9. 🖥️ GRADIO UI (FRONTEND)

In [18]:
with gr.Blocks(title="DocAI", theme=gr.themes.Soft(primary_hue="blue")) as demo:

    gr.Markdown("# 📄 DocAI — HuggingFace RAG System")

    with gr.Row():

        with gr.Column(scale=1):
            pdf_input = gr.File(label="Upload PDF", file_types=[".pdf"], type="filepath")

            with gr.Accordion("Settings", open=False):
                model_dd = gr.Dropdown(
                    choices=list(MODEL_MAP.keys()),
                    value="Qwen2.5-7B",
                    label="Model"
                )

                chunk_sz = gr.Slider(200, 2000, value=1000, step=100, label="Chunk Size")
                chunk_ov = gr.Slider(0, 500, value=150, step=25, label="Chunk Overlap")

            process_btn = gr.Button("Process PDF", variant="primary")
            status_box = gr.Textbox(label="Status", lines=8)

        with gr.Column(scale=2):

            gr.ChatInterface(
                fn=answer_question,
                chatbot=gr.Chatbot(height=450),
                textbox=gr.Textbox(placeholder="Ask anything from PDF..."),
                examples=[
                    "What is this document about?",
                    "Summarize key points",
                    "What are the conclusions?"
                ],
            )

    process_btn.click(
        fn=process_pdf,
        inputs=[pdf_input, chunk_sz, chunk_ov, model_dd],
        outputs=[status_box]
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://267681e355c376ea45.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
